In [ ]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, MaxPooling1D
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except Exception:
    TENSORFLOW_AVAILABLE = False

In [22]:
BASE_DIR = os.getcwd()
CLOUD_PATH = os.path.join("/content/drive/My Drive/Colab Notebooks", "Cloud_Dataset.csv")
PRIMARY_PATH = os.path.join("/content/drive/My Drive/Colab Notebooks", "Primary Dataset.xlsx")
OUTPUT_DIR = os.path.join(BASE_DIR, "chapter4_outputs")
CHART_DIR = os.path.join(OUTPUT_DIR, "charts")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")

os.makedirs(CHART_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

In [23]:
def save_chart(filename):
    """Save the active matplotlib chart."""
    path = os.path.join(CHART_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    return path


def mape(y_true, y_pred):
    """Mean Absolute Percentage Error with zero protection."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_model(name, y_true, y_pred):
    """Return model evaluation metrics."""
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


def make_sequences(values, lookback=12):
    """Create supervised sequences for LSTM-style models."""
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i, 0])
    return np.array(X), np.array(y)


def clean_col(name):
    """Shorten long survey column names into safer labels."""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return name


def cramers_v(confusion_matrix):
    """Cramer's V effect size for chi-square tables."""
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1)))

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
print(f"Attempting to read Cloud Dataset from: {CLOUD_PATH}")
cloud = pd.read_csv(CLOUD_PATH)
primary = pd.read_excel(PRIMARY_PATH)
primary.columns = [clean_col(c) for c in primary.columns]

print("Secondary dataset shape:", cloud.shape)
print("Primary dataset shape:", primary.shape)

Attempting to read Cloud Dataset from: /content/drive/My Drive/Colab Notebooks/Cloud_Dataset.csv
Secondary dataset shape: (1000, 16)
Primary dataset shape: (82, 25)


In [26]:
cloud["timestamp"] = pd.to_datetime(cloud["timestamp"], errors="coerce")
cloud = cloud.sort_values("timestamp").reset_index(drop=True)

numeric_cols = cloud.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    cloud[col] = cloud[col].interpolate(method="linear").fillna(method="bfill").fillna(method="ffill")

cloud["hour"] = cloud["timestamp"].dt.hour
cloud["dayofweek"] = cloud["timestamp"].dt.dayofweek
cloud["is_scale_up"] = (cloud["target"].astype(str).str.lower().str.contains("up")).astype(int)
cloud["is_scale_down"] = (cloud["target"].astype(str).str.lower().str.contains("down")).astype(int)
cloud["is_no_action"] = (cloud["target"].astype(str).str.lower().str.contains("no")).astype(int)

# Save basic descriptive statistics
cloud.describe(include="all").to_csv(os.path.join(TABLE_DIR, "secondary_dataset_descriptive_statistics.csv"))
cloud.isna().sum().to_csv(os.path.join(TABLE_DIR, "secondary_dataset_missing_values.csv"))


In [27]:
plt.figure(figsize=(12, 5))
plt.plot(cloud["timestamp"], cloud["cpu_usage"])
plt.title("CPU Usage Over Time")
plt.xlabel("Timestamp")
plt.ylabel("CPU Usage (%)")
save_chart("figure_4_1_cpu_usage_over_time.png")

plt.figure(figsize=(12, 5))
plt.plot(cloud["timestamp"], cloud["memory_usage"])
plt.title("Memory Usage Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Memory Usage (%)")
save_chart("figure_4_2_memory_usage_over_time.png")

plt.figure(figsize=(8, 5))
cloud.groupby("hour")["cpu_usage"].mean().plot(kind="bar")
plt.title("Average CPU Usage by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Average CPU Usage (%)")
save_chart("figure_4_3_average_cpu_by_hour.png")

plt.figure(figsize=(8, 5))
cloud["target"].value_counts().plot(kind="bar")
plt.title("Distribution of Scaling Actions")
plt.xlabel("Scaling Action")
plt.ylabel("Frequency")
save_chart("figure_4_4_scaling_action_distribution.png")

corr_cols = ["cpu_usage", "memory_usage", "net_io", "disk_io", "latency_ms", "throughput", "cost", "utilization"]
corr = cloud[corr_cols].corr()
corr.to_csv(os.path.join(TABLE_DIR, "secondary_dataset_correlation_matrix.csv"))
plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label="Correlation")
plt.title("Correlation Matrix of Cloud Resource Variables")
save_chart("figure_4_5_secondary_correlation_matrix.png")

plt.figure(figsize=(8, 5))
cloud.boxplot(column="cpu_usage", by="cloud_provider")
plt.title("CPU Usage by Cloud Provider")
plt.suptitle("")
plt.xlabel("Cloud Provider")
plt.ylabel("CPU Usage (%)")
save_chart("figure_4_6_cpu_by_cloud_provider.png")

'/content/chapter4_outputs/charts/figure_4_6_cpu_by_cloud_provider.png'

<Figure size 800x500 with 0 Axes>

In [28]:
stationarity_results = []
if STATSMODELS_AVAILABLE:
    for col in ["cpu_usage", "memory_usage", "utilization"]:
        result = adfuller(cloud[col].dropna())
        stationarity_results.append({
            "Variable": col,
            "ADF Statistic": result[0],
            "p-value": result[1],
            "Stationary at 5%": result[1] < 0.05
        })
    pd.DataFrame(stationarity_results).to_csv(os.path.join(TABLE_DIR, "adf_stationarity_tests.csv"), index=False)

In [29]:
series = cloud[["cpu_usage"]].values
train_size = int(len(series) * 0.70)
val_size = int(len(series) * 0.15)
train = series[:train_size]
val = series[train_size:train_size + val_size]
test = series[train_size + val_size:]

model_results = []
forecast_df = pd.DataFrame({"actual_cpu": test.flatten()})

# 7.1 Naive baseline
naive_pred = np.repeat(train[-1], len(test))
model_results.append(evaluate_model("Naive Baseline", test.flatten(), naive_pred.flatten()))
forecast_df["naive_baseline"] = naive_pred.flatten()

# 7.2 ARIMA
if STATSMODELS_AVAILABLE:
    try:
        arima_model = ARIMA(train.flatten(), order=(5, 1, 2))
        arima_fit = arima_model.fit()
        arima_pred = arima_fit.forecast(steps=len(test))
        model_results.append(evaluate_model("ARIMA", test.flatten(), arima_pred))
        forecast_df["arima"] = arima_pred
    except Exception as e:
        print("ARIMA failed:", e)

# 7.3 Random Forest benchmark using lag features
lookback = 12
cpu_df = pd.DataFrame({"cpu_usage": cloud["cpu_usage"]})
for lag in range(1, lookback + 1):
    cpu_df[f"lag_{lag}"] = cpu_df["cpu_usage"].shift(lag)
cpu_df = cpu_df.dropna()
X = cpu_df[[f"lag_{lag}" for lag in range(1, lookback + 1)]]
y = cpu_df["cpu_usage"]
X_train, X_test = X.iloc[:int(len(X)*0.85)], X.iloc[int(len(X)*0.85):]
y_train, y_test = y.iloc[:int(len(y)*0.85)], y.iloc[int(len(y)*0.85):]
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
model_results.append(evaluate_model("Random Forest Lag Benchmark", y_test, rf_pred))

# 7.4 LSTM and CNN-LSTM
if TENSORFLOW_AVAILABLE:
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(series)
    X_seq, y_seq = make_sequences(scaled, lookback=lookback)
    split1 = int(len(X_seq) * 0.70)
    split2 = int(len(X_seq) * 0.85)
    X_train, y_train = X_seq[:split1], y_seq[:split1]
    X_val, y_val = X_seq[split1:split2], y_seq[split1:split2]
    X_test, y_test = X_seq[split2:], y_seq[split2:]

    early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

    lstm_model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.2),
        Dense(1)
    ])
    lstm_model.compile(optimizer="adam", loss="mse")
    lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=80, batch_size=32, callbacks=[early_stop], verbose=0)
    lstm_pred_scaled = lstm_model.predict(X_test, verbose=0)
    lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    model_results.append(evaluate_model("LSTM", y_test_inv, lstm_pred))

    cnn_lstm_model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation="relu", input_shape=(X_train.shape[1], X_train.shape[2])),
        MaxPooling1D(pool_size=2),
        LSTM(64),
        Dropout(0.2),
        Dense(1)
    ])
    cnn_lstm_model.compile(optimizer="adam", loss="mse")
    cnn_lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=80, batch_size=32, callbacks=[early_stop], verbose=0)
    cnn_lstm_pred_scaled = cnn_lstm_model.predict(X_test, verbose=0)
    cnn_lstm_pred = scaler.inverse_transform(cnn_lstm_pred_scaled).flatten()
    model_results.append(evaluate_model("CNN-LSTM", y_test_inv, cnn_lstm_pred))

    plt.figure(figsize=(12, 5))
    plt.plot(y_test_inv, label="Actual CPU")
    plt.plot(lstm_pred, label="LSTM Forecast")
    plt.plot(cnn_lstm_pred, label="CNN-LSTM Forecast")
    plt.title("Deep Learning Forecasts Compared with Actual CPU Usage")
    plt.xlabel("Test Observation")
    plt.ylabel("CPU Usage (%)")
    plt.legend()
    save_chart("figure_4_7_lstm_cnn_lstm_forecast.png")
else:
    print("TensorFlow is not available. LSTM and CNN-LSTM sections were skipped.")

# Save model results
model_results_df = pd.DataFrame(model_results).sort_values("RMSE")
model_results_df.to_csv(os.path.join(TABLE_DIR, "forecasting_model_comparison.csv"), index=False)
forecast_df.to_csv(os.path.join(TABLE_DIR, "forecast_values_arima_baseline.csv"), index=False)

plt.figure(figsize=(9, 5))
model_results_df.set_index("Model")["RMSE"].plot(kind="bar")
plt.title("Forecasting Model Comparison by RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
save_chart("figure_4_8_model_comparison_rmse.png")

'/content/chapter4_outputs/charts/figure_4_8_model_comparison_rmse.png'

In [30]:
primary_clean = primary.copy()
primary_clean.to_csv(os.path.join(TABLE_DIR, "primary_dataset_cleaned.csv"), index=False)

# Identify numeric Likert-style questions already stored as numbers
likert_cols = primary_clean.select_dtypes(include=[np.number]).columns.tolist()
primary_clean[likert_cols].describe().to_csv(os.path.join(TABLE_DIR, "primary_likert_descriptive_statistics.csv"))

# Chart each numeric Likert question
for i, col in enumerate(likert_cols, start=1):
    plt.figure(figsize=(8, 5))
    primary_clean[col].value_counts().sort_index().plot(kind="bar")
    plt.title(f"Survey Response Distribution: Q{i}")
    plt.xlabel("Likert Rating")
    plt.ylabel("Frequency")
    save_chart(f"figure_4_9_{i}_survey_likert_distribution.png")

# Correlation matrix for numeric survey variables
if len(likert_cols) >= 2:
    primary_corr = primary_clean[likert_cols].corr(method="spearman")
    primary_corr.to_csv(os.path.join(TABLE_DIR, "primary_spearman_correlation_matrix.csv"))
    plt.figure(figsize=(9, 7))
    plt.imshow(primary_corr, aspect="auto")
    plt.xticks(range(len(primary_corr.columns)), [f"Q{i+1}" for i in range(len(primary_corr.columns))], rotation=45, ha="right")
    plt.yticks(range(len(primary_corr.columns)), [f"Q{i+1}" for i in range(len(primary_corr.columns))])
    plt.colorbar(label="Spearman Correlation")
    plt.title("Primary Survey Spearman Correlation Matrix")
    save_chart("figure_4_10_primary_spearman_correlation_matrix.png")

# Reliability: Cronbach alpha for numeric Likert questions
def cronbach_alpha(df):
    df = df.dropna()
    k = df.shape[1]
    if k < 2:
        return np.nan
    item_vars = df.var(axis=0, ddof=1)
    total_var = df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

alpha = cronbach_alpha(primary_clean[likert_cols])
pd.DataFrame([{"Cronbach Alpha": alpha, "Number of Items": len(likert_cols)}]).to_csv(
    os.path.join(TABLE_DIR, "primary_cronbach_alpha.csv"), index=False
)

# Regression: predict overall satisfaction from other numeric survey variables
satisfaction_cols = [c for c in likert_cols if "satisfied" in c.lower() or "satisfaction" in c.lower()]
if satisfaction_cols:
    target_col = satisfaction_cols[0]
    predictors = [c for c in likert_cols if c != target_col]
    reg_df = primary_clean[[target_col] + predictors].dropna()
    X = reg_df[predictors]
    y = reg_df[target_col]
    linreg = LinearRegression()
    linreg.fit(X, y)
    y_pred = linreg.predict(X)
    regression_summary = pd.DataFrame({
        "Predictor": predictors,
        "Coefficient": linreg.coef_
    })
    regression_summary.loc[len(regression_summary)] = ["Intercept", linreg.intercept_]
    regression_summary.loc[len(regression_summary)] = ["R2", r2_score(y, y_pred)]
    regression_summary.to_csv(os.path.join(TABLE_DIR, "primary_regression_satisfaction_model.csv"), index=False)

    plt.figure(figsize=(7, 5))
    plt.scatter(y, y_pred)
    plt.title("Actual vs Predicted Online Service Satisfaction")
    plt.xlabel("Actual Satisfaction")
    plt.ylabel("Predicted Satisfaction")
    save_chart("figure_4_11_primary_regression_actual_vs_predicted.png")

# Spearman tests against satisfaction
spearman_results = []
if satisfaction_cols:
    target_col = satisfaction_cols[0]
    for col in [c for c in likert_cols if c != target_col]:
        rho, p = stats.spearmanr(primary_clean[col], primary_clean[target_col], nan_policy="omit")
        spearman_results.append({
            "Variable": col,
            "Compared With": target_col,
            "Spearman rho": rho,
            "p-value": p,
            "Significant at 5%": p < 0.05
        })
    pd.DataFrame(spearman_results).to_csv(os.path.join(TABLE_DIR, "primary_spearman_tests_against_satisfaction.csv"), index=False)

# Chi-square tests for categorical variables
cat_cols = primary_clean.select_dtypes(include=["object"]).columns.tolist()
chi_results = []
important_question = None
for c in cat_cols:
    if "predict" in c.lower() and "busy" in c.lower():
        important_question = c
        break

if important_question:
    for col in cat_cols:
        if col != important_question and primary_clean[col].nunique(dropna=True) > 1 and primary_clean[col].nunique(dropna=True) <= 12:
            table = pd.crosstab(primary_clean[col], primary_clean[important_question])
            if table.shape[0] > 1 and table.shape[1] > 1:
                chi2, p, dof, expected = stats.chi2_contingency(table)
                chi_results.append({
                    "Variable": col,
                    "Compared With": important_question,
                    "Chi-square": chi2,
                    "p-value": p,
                    "Degrees of Freedom": dof,
                    "Cramers V": cramers_v(table),
                    "Significant at 5%": p < 0.05
                })
    pd.DataFrame(chi_results).to_csv(os.path.join(TABLE_DIR, "primary_chi_square_tests.csv"), index=False)

# Frequency tables for categorical survey questions
with pd.ExcelWriter(os.path.join(TABLE_DIR, "primary_categorical_frequency_tables.xlsx")) as writer:
    for i, col in enumerate(cat_cols):
        freq = primary_clean[col].value_counts(dropna=False).reset_index()
        freq.columns = ["Response", "Frequency"]
        sheet = f"Q{i+1}"[:31]
        freq.to_excel(writer, sheet_name=sheet, index=False)

# Qualitative open-ended categorisation
open_cols = [c for c in primary_clean.columns if "anything else" in c.lower() or "share" in c.lower()]
qual_rows = []
if open_cols:
    open_col = open_cols[0]
    comments = primary_clean[open_col].dropna().astype(str)
    for comment in comments:
        text = comment.lower()
        if any(word in text for word in ["slow", "bad", "problem", "load", "down", "unavailable"]):
            theme = "Performance and availability concerns"
        elif any(word in text for word in ["authority", "control", "regulation", "responsible"]):
            theme = "Governance and responsible management"
        elif any(word in text for word in ["good", "satisfied", "reliable"]):
            theme = "Positive service experience"
        else:
            theme = "General user experience"
        qual_rows.append({"Comment": comment, "Theme": theme})

    qual_df = pd.DataFrame(qual_rows)
    qual_df.to_csv(os.path.join(TABLE_DIR, "primary_qualitative_comment_themes.csv"), index=False)

    if not qual_df.empty:
        plt.figure(figsize=(8, 5))
        qual_df["Theme"].value_counts().plot(kind="bar")
        plt.title("Themes from Open-Ended Survey Responses")
        plt.xlabel("Theme")
        plt.ylabel("Frequency")
        save_chart("figure_4_12_qualitative_comment_themes.png")


In [31]:
summary_path = os.path.join(OUTPUT_DIR, "chapter4_results_summary.xlsx")
with pd.ExcelWriter(summary_path) as writer:
    cloud.describe().to_excel(writer, sheet_name="Secondary Descriptives")
    corr.to_excel(writer, sheet_name="Secondary Correlation")
    model_results_df.to_excel(writer, sheet_name="Model Comparison", index=False)
    primary_clean[likert_cols].describe().to_excel(writer, sheet_name="Primary Descriptives")
    if len(likert_cols) >= 2:
        primary_corr.to_excel(writer, sheet_name="Primary Correlation")
    pd.DataFrame([{"Cronbach Alpha": alpha, "Number of Items": len(likert_cols)}]).to_excel(writer, sheet_name="Cronbach Alpha", index=False)
    if spearman_results:
        pd.DataFrame(spearman_results).to_excel(writer, sheet_name="Spearman Tests", index=False)
    if chi_results:
        pd.DataFrame(chi_results).to_excel(writer, sheet_name="Chi Square Tests", index=False)

print("\nAnalysis complete.")
print("Charts saved to:", CHART_DIR)
print("Tables saved to:", TABLE_DIR)
print("Summary workbook saved to:", summary_path)



Analysis complete.
Charts saved to: /content/chapter4_outputs/charts
Tables saved to: /content/chapter4_outputs/tables
Summary workbook saved to: /content/chapter4_outputs/chapter4_results_summary.xlsx
